<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">📘 Batch, micro-batch a streaming - Northwind </h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">V nadväznosti na Seminár 8 - Zadanie 1a s MovieLens spracujeme údaje o objednávkach. Tentoraz si jednotlivé kroky naprogramujete samostatne od načítania CSV po porovnanie výsledkov.</p>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Z kurzu si stiahnite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders.csv</code> (stĺpce <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderid</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">customerid</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">employeeid</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderdate</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">shipperid</code>). Súbor uložte do rovnakého priečinka ako notebook. Knižnice a PostgreSQL už máte pripravené.</p>

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 1 – Načítanie a časová distribúcia objednávok</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Načítajte súbor <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders.csv</code> a zmeňte <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderdate</code> na dátum. Zistite časový rozsah a počet objednávok podľa roku a mesiaca. Rozloženie zobrazte v grafe.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Pomôcka:</strong> <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">pd.read_csv(..., parse_dates=["orderdate"])</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">.dt.year</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">.dt.to_period("M")</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">value_counts()</code>.</div>

In [ ]:
# Vaše riešenie
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path("orders.csv")
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV sa nenašlo: {CSV_PATH}. Upravte CSV_PATH.")

orders = pd.read_csv(CSV_PATH, parse_dates=["orderdate"])

required = {"orderid", "customerid", "employeeid", "orderdate", "shipperid"}
missing = required - set(orders.columns)
if missing:
    raise ValueError(f"V CSV chýbajú stĺpce: {sorted(missing)}")

print("Počet objednávok:", len(orders))
print("Najstaršia objednávka:", orders["orderdate"].min())
print("Najnovšia objednávka:", orders["orderdate"].max())

orders_by_year = orders["orderdate"].dt.year.value_counts().sort_index()
display(orders_by_year)

orders_by_month = orders["orderdate"].dt.to_period("M").value_counts().sort_index()
display(orders_by_month)

orders_by_month.plot(
    kind="bar",
    figsize=(10, 3.5),
    color="#3577d4",
    title="Počet objednávok podľa mesiaca"
)
plt.xlabel("Mesiac")
plt.ylabel("Počet objednávok")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 2 – Historické a novšie objednávky</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Podľa časového rozloženia zvoľte <strong>hranice 1. januára 1997</strong>. Staršie objednávky budú historický batch, objednávky od tohto dátumu využijeme na simuláciu príchodu udalostí.</p>
<ul style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.7; margin-top: 6px; padding-left: 24px;"><li style="margin-bottom: 5px;"><code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">historical_orders</code> budú objednávky pred hranicou,</li><li style="margin-bottom: 5px;"><code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">new_orders</code> naopak objednávky od hranice (vrátane),</li><li style="margin-bottom: 5px;">overte, že obe skupiny spolu obsahujú všetky objednávky.</li></ul>
<div style="margin-top: 12px; background-color: #fefce8; border-left: 4px solid #f59e0b; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">⚠️ <strong>Upozornenie: </strong> „Novšie“ objednávky sú stále <strong>historické záznamy</strong>. V nasledujúcich úlohách budeme ich príchod len simulovať.</div>

In [ ]:
# Vaše riešenie
CUTOFF = pd.Timestamp("1997-01-01")

historical_orders = orders[orders["orderdate"] < CUTOFF].copy()
new_orders = orders[orders["orderdate"] >= CUTOFF].copy()

print("Historické objednávky:", len(historical_orders))
print("Novšie objednávky:", len(new_orders))

assert len(historical_orders) + len(new_orders) == len(orders)
print("Spolu:", len(historical_orders) + len(new_orders), "z", len(orders))

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 3 – Raw ingestion do PostgreSQL</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Vytvorte databázu <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">Northwind</code>, tabuľku <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders</code> s primárnym kľúčom <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderid</code> a nahrajte všetky záznamy z CSV. Predpokladáme, že PostgreSQL a balíky už fungujú inštaláciu ani test verzie preto nemusíte riešiť.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Pomôcka:</strong> najskôr sa pripojte do databázy <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">postgres</code>, vytvorte <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders_practice</code> s <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">AUTOCOMMIT</code>, potom zmeňte pripojenie. Na opakovateľný import použite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">DROP TABLE IF EXISTS</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">to_sql(..., if_exists="append", index=False)</code>.</div>
<div style="margin-top: 12px; background-color: #fefce8; border-left: 4px solid #f59e0b; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">⚠️ <strong> Upozornenie:</strong> Pri opakovanom spustení sa obsah tabuľky <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orders</code> vymaže a znovu nahrá. Nepoužívajte názov databázy, ktorá už obsahuje vaše vlastné dáta.</div>

In [ ]:
# Vaše riešenie
from sqlalchemy import create_engine, text

PG_USER, PG_PASSWORD = "postgres", "Student"   
PG_HOST, PG_PORT = "localhost", 5432
DB_NAME = "orders_practice"

def pg_url(db):
    return f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{db}"

admin_engine = create_engine(pg_url("postgres"))
with admin_engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": DB_NAME}
    ).scalar()
    if not exists:
        conn.execute(text(f'CREATE DATABASE "{DB_NAME}"'))
admin_engine.dispose()

engine = create_engine(pg_url(DB_NAME))

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS orders"))
    conn.execute(text("""
        CREATE TABLE orders (
            orderid    INT PRIMARY KEY,
            customerid INT,
            employeeid INT,
            orderdate  DATE,
            shipperid  INT
        )
    """))

orders.to_sql("orders", engine, if_exists="append", index=False)

with engine.connect() as conn:
    db_count = conn.execute(text("SELECT COUNT(*) FROM orders")).scalar()
assert db_count == len(orders)

print("Tabuľka orders bola vytvorená a naplnená.")
print("Riadkov v databáze:", db_count)

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 4 – Batch processing</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Z historických objednávok vypočítajte <strong>počet objednávok</strong>, <strong>počet zákazníkov</strong> a <strong>percento objednávok s <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">shipperid = 3</code></strong>. Výsledok získame jednorazovo za celú historickú skupinu.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong> Poznámka:</strong> Percento vypočítate ako <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">(počet objednávok dopravcu 3 / počet všetkých objednávok) * 100</code>. Výraz <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">(df["shipperid"] == 3).sum()</code> spočíta vyhovujúce riadky.</div>

In [ ]:
# Vaše riešenie
hist_count = len(historical_orders)
hist_customers = historical_orders["customerid"].nunique()
hist_shipper3 = (historical_orders["shipperid"] == 3).sum()
hist_shipper3_pct = hist_shipper3 / hist_count * 100

print("Objednávok:", hist_count)
print("Zákazníkov:", hist_customers)
print("Dopravca 3 (%):", round(hist_shipper3_pct, 2))

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 5 – Spoločná vzorka pre tri prístupy</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Na porovnanie musí každý prístup dostať <strong>rovnaké objednávky v rovnakom poradí</strong>. Z PostgreSQL vyberte novšie objednávky a zoraďte ich podľa <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderdate</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">orderid</code>.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong> Poznámka:</strong> Použite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">text()</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">engine.connect()</code> a pomenované parametre SQL <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">:cutoff</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">:limit</code>. V tomto datasete je po hranici 44 objednávok, preto postačuje <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">SAMPLE_SIZE = 44</code>.</div>

In [ ]:
# Vaše riešenie
SAMPLE_SIZE = 44

with engine.connect() as conn:
    event_sample = pd.read_sql(
        text("""
            SELECT orderid, customerid, employeeid, orderdate, shipperid
            FROM orders
            WHERE orderdate >= :cutoff
            ORDER BY orderdate, orderid
            LIMIT :limit
        """),
        conn,
        params={"cutoff": CUTOFF.date(), "limit": SAMPLE_SIZE},
        parse_dates=["orderdate"]
    )

if event_sample.empty:
    raise ValueError("Po hranici nie sú dostupné objednávky. Skontrolujte dáta alebo CUTOFF.")

print("Objednávok v simulácii:", len(event_sample))
print("Časový rozsah:", event_sample["orderdate"].min().date(), "–",
      event_sample["orderdate"].max().date())

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 6 – Simulácia micro-batchingu a streamingu</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Historické objednávky budeme prehrávať postupne. <strong>Streaming</strong> aktualizuje podiel dopravcu 3 po každej objednávke, <strong>micro-batch</strong> po uzavretí krátkeho časového okna. Batch poskytne výsledok až po príchode celej vzorky.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Poznámka:</strong> Pre tento malý dataset nastavíme <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">EVENT_DELAY_SEC = 0.04</code> (40 ms medzi objednávkami) a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">MICRO_WINDOW_SEC = 0.30</code> (približne 300 ms na dávku). Skutočné časy závisia od výkonu počítača.</div>

<h3 style="margin: 20px 0 10px; font-size: 17px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827;">Krok A – Nastavenia a priebežný stav</h3>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Pripravte časové parametre, počítadlá a zoznamy na zaznamenávanie priebežných výsledkov. Pri micro-batchingu potrebujeme navyše premenné pre neuzavretú dávku.</p>

In [ ]:
# Vaše riešenie
import time

EVENT_DELAY_SEC = 0.04   
MICRO_WINDOW_SEC = 0.30   

stream_running_count = 0    
stream_running_sum = 0      
stream_updates = []         

micro_running_count = 0
micro_running_sum = 0
micro_updates = []         

micro_pending_start = 0

<h3 style="margin: 20px 0 10px; font-size: 17px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827;">Krok B – Spracovanie udalostí</h3>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Prejdite objednávky postupne. Pri streamingu priebežne zapisujte <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">(čas, počet, percento)</code>. Pri micro-batchingu spracujte nazbierané objednávky až po uplynutí okna.</p>

In [ ]:
# Vaše riešenie
assert stream_running_count == 0, "Pred opakovaným spustením znovu spustite Krok A."

start_time = time.perf_counter()
last_flush = start_time

for i in range(len(event_sample)):
    time.sleep(EVENT_DELAY_SEC)       
    now = time.perf_counter()
    order = event_sample.iloc[i]       
    is_shipper3 = int(order["shipperid"] == 3)

    stream_running_count += 1
    stream_running_sum += is_shipper3
    stream_updates.append((now - start_time, stream_running_count,
                           stream_running_sum / stream_running_count * 100))

    if now - last_flush >= MICRO_WINDOW_SEC:
        batch = event_sample.iloc[micro_pending_start:i + 1]
        micro_running_count += len(batch)
        micro_running_sum += int((batch["shipperid"] == 3).sum())
        micro_updates.append((now - start_time, micro_running_count,
                              micro_running_sum / micro_running_count * 100))
        micro_pending_start = i + 1    
        last_flush = now

end_time = time.perf_counter()

print("Spracovaných objednávok:", stream_running_count)
print("Uzavretých micro-batch dávok:", len(micro_updates),
      "| čaká na spracovanie:", len(event_sample) - micro_pending_start)

<h3 style="margin: 20px 0 10px; font-size: 17px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827;">Krok C – Posledná dávka a batch</h3>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Nezabudnite spracovať poslednú nedokončenú micro-batch dávku. Nakoniec vypočítajte jednorazový batch výsledok pre tú istú vzorku a overte rovnosť výsledných percent.</p>

In [ ]:
# Vaše riešenie
if micro_pending_start < len(event_sample):
    batch = event_sample.iloc[micro_pending_start:]
    micro_running_count += len(batch)
    micro_running_sum += int((batch["shipperid"] == 3).sum())
    micro_updates.append((end_time - start_time, micro_running_count,
                          micro_running_sum / micro_running_count * 100))
    micro_pending_start = len(event_sample)

batch_start = time.perf_counter()
batch_count = len(event_sample)
batch_pct = (event_sample["shipperid"] == 3).sum() / batch_count * 100
batch_ready_sec = (end_time - start_time) + (time.perf_counter() - batch_start)

micro_pct = micro_running_sum / micro_running_count * 100
stream_pct = stream_running_sum / stream_running_count * 100

print("Počet objednávok:", batch_count)
print("Batch (%):", round(batch_pct, 2))
print("Micro-batch (%):", round(micro_pct, 2))
print("Streaming (%):", round(stream_pct, 2))

assert stream_running_count == micro_running_count == batch_count
assert abs(batch_pct - micro_pct) < 1e-9 and abs(batch_pct - stream_pct) < 1e-9
print("Výsledné percentá sa zhodujú.")

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 7 – Grafy priebežných výsledkov</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Vytvorte <strong>tri samostatné grafy</strong> pod sebou. V prvom zobrazte jednorazový batch bod, v druhom schodovitý priebeh micro-batchingu a v treťom streamingovú krivku. Použite spoločné osi, aby boli percentá porovnateľné.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong>Poznámka:</strong> Použite <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">plt.subplots(3, 1, sharex=True, sharey=True)</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">scatter()</code>, <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">step(where="post")</code> a <code style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">plot()</code>.</div>

In [ ]:
# Vaše riešenie
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, sharey=True)

axes[0].scatter([batch_ready_sec], [batch_pct], color="#3577d4", s=100, zorder=5)
axes[0].set_title("Batch – výsledok až na konci")

axes[1].step(
    [x[0] for x in micro_updates],
    [x[2] for x in micro_updates],
    where="post", color="#159572", linewidth=2
)
axes[1].set_title("Micro-batch – výsledok po každej dávke")

axes[2].plot(
    [x[0] for x in stream_updates],
    [x[2] for x in stream_updates],
    color="#c88730", linewidth=1.8
)
axes[2].set_title("Streaming – výsledok po každej udalosti")

for ax in axes:
    ax.set_ylabel("Dopravca 3 (%)")
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[2].set_xlabel("Čas od začiatku simulácie (s)")
plt.tight_layout(h_pad=2)
plt.show()

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">💻 Úloha 8 – Frekvencia aktualizácií a prvý výsledok</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">Na troch grafoch znázornite, koľkokrát sa priebežný výsledok aktualizoval. Potom vytvorte jednoduchú tabuľku, ktorá porovná <strong>čas prvého výsledku</strong>.</p>
<div style="margin-top: 12px; background-color: #f9fafb; border-left: 4px solid #60a5fa; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">💡 <strong> Poznámka:</strong> Pri počte aktualizácií nie je vhodné zdieľať os Y: batch má jeden výsledok, streaming jeden po každej udalosti. Os X zdieľajte.</div>

In [ ]:
# Vaše riešenie
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].step([0, batch_ready_sec], [0, 1], where="post", color="#3577d4", linewidth=2)
axes[0].scatter([batch_ready_sec], [1], color="#3577d4", s=70, zorder=5)
axes[0].set_title("Batch")

axes[1].step(
    [0] + [x[0] for x in micro_updates],
    [0] + list(range(1, len(micro_updates) + 1)),
    where="post", color="#159572", linewidth=2
)
axes[1].set_title("Micro-batch")

axes[2].step(
    [0] + [x[0] for x in stream_updates],
    [0] + list(range(1, len(stream_updates) + 1)),
    where="post", color="#c88730", linewidth=1.8
)
axes[2].set_title("Event streaming")

for ax in axes:
    ax.set_ylabel("Počet aktualizácií")
    ax.grid(alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

axes[2].set_xlabel("Čas od začiatku simulácie (s)")
plt.tight_layout(h_pad=2)
plt.show()

results = pd.DataFrame({
    "Prístup": ["Batch", "Micro-batch", "Event streaming"],
    "Prvý výsledok (s)": [batch_ready_sec, micro_updates[0][0], stream_updates[0][0]],
    "Aktualizácií": [1, len(micro_updates), len(stream_updates)],
})
print(results.to_string(index=False, formatters={"Prvý výsledok (s)": "{:.3f}".format}))
print("\nPrvý výsledok najskôr:", results.loc[results["Prvý výsledok (s)"].idxmin(), "Prístup"])

<h2 style="margin: 24px 0 16px; font-size: 20px; font-weight: 600; font-family: 'Segoe UI', Roboto, sans-serif; color: #111827; border-bottom: 2px solid #e5e7eb; padding-bottom: 6px;">
  📘 Kontrola Git repozitára po 8. týždni
</h2>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">
  Po dokončení všetkých úloh z 8. týždňa skontrolujte, či máte vo svojom Git
  repozitári vytvorenú samostatnú vetvu
  <span style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">week-08</span>
  a či sa v nej nachádza vypracovaný notebook vytvorený počas cvičenia.
</p>
<p style="font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.65; margin: 8px 0 12px;">
  Obsah vetvy
  <span style="background-color: #e5e7eb; color: #111827; font-family: monospace; padding: 2px 5px; border-radius: 4px;">week-08</span>
  by mal mať približne nasledujúcu štruktúru:
</p>
<pre style="background: #f9fafb; color: #111827; padding: 14px; border-radius: 7px; overflow: auto; font-size: 13px; line-height: 1.7; font-family: ui-monospace, SFMono-Regular, Menlo, monospace;">week-08
│
├── ratings.csv
├── orders.csv
│
├── batch_streaming.ipynb
└── batch_streaming_orders.ipynb
</pre>
<div style="margin-top: 12px; background-color: #fefce8; border-left: 4px solid #eab308; padding: 12px; border-radius: 4px; font-family: 'Segoe UI', Roboto, sans-serif; color: #1f2937; line-height: 1.6;">
  ⚠️ <strong>Kontrola GitHubu:</strong> Skontrolujte, či sa všetky uvedené súbory
  nachádzajú aj vo vzdialenom repozitári na GitHube. Ak ich vidíte iba lokálne
  vo VS Code, zmeny ešte neboli úspešne odoslané pomocou
  <strong>commit</strong> a <strong>push</strong>.
</div>